# MCTS vs ID3 — PopOut
This notebook:
1. Compiles the C game engine
2. Generates training data by running MCTS self-play
3. Trains an ID3 decision tree on that data
4. Runs a match series: **MCTS (`#`) vs ID3 (`@`)**
5. Reports win/loss/draw statistics

**Required files in the same directory as this notebook:**
- `popout.c`
- `MonteCarlo.c`
- `Stack.c`
- `mcts_driver.c`
- `ID3_DECISION_TREES.py`

## 1 — Compile the C engine

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    ["gcc", "-O2", "-o", "mcts", "mcts_driver.c", "-lm"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("COMPILATION FAILED")
    print(result.stderr)
    sys.exit(1)
else:
    print("Compiled OK →", os.path.abspath("mcts"))

## 2 — Python helpers

In [ ]:
import subprocess
import numpy as np
import pandas as pd
import copy

MAX_X, MAX_Y = 7, 6

# ── board utilities ──────────────────────────────────────────────────

def empty_board():
    """board[x][y], x=col 0..6, y=row 0..5 (0=top)"""
    return [['_'] * MAX_Y for _ in range(MAX_X)]

def board_to_stdin(board, player, iterations=500):
    """Format a board as the stdin string expected by mcts_driver."""
    lines = [player]
    for y in range(MAX_Y):
        lines.append(''.join(board[x][y] for x in range(MAX_X)))
    lines.append(str(iterations))
    return '\n'.join(lines) + '\n'

def board_to_str(board):
    """Flatten board to a 42-char string (row-major, top first)."""
    return ''.join(board[x][y] for y in range(MAX_Y) for x in range(MAX_X))

def ask_mcts(board, player, iterations=500):
    """Call the compiled MCTS binary and return (col, option) or ('draw', 0)."""
    stdin = board_to_stdin(board, player, iterations)
    r = subprocess.run(
        ["./mcts", "--quiet"],
        input=stdin, capture_output=True, text=True, timeout=30
    )
    if r.returncode != 0:
        raise RuntimeError(f"mcts binary error:\n{r.stderr}")
    out = r.stdout.strip()
    if out == "draw":
        return 0, 0
    # "column N insert" or "column N pop"
    parts = out.split()
    col = int(parts[1])
    opt = 0 if parts[2] == "insert" else 1
    return col, opt

# ── in-Python game logic (mirrors C logic) ───────────────────────────

def do_insert(board, col, player):
    """Insert player's piece into col (1-based). Returns True on win."""
    b = copy.deepcopy(board)
    y = 0
    while y < MAX_Y and b[col-1][y] == '_':
        y += 1
    place = y - 1 if y != 0 else 0
    b[col-1][place] = player
    return b, check_win_insert(b, col-1, place, player)

def do_pop(board, col, player):
    """Pop bottom piece of col (1-based). Returns (new_board, win, self_loss)."""
    b = copy.deepcopy(board)
    b[col-1][MAX_Y-1] = '_'
    for i in range(MAX_Y-1, 0, -1):
        b[col-1][i] = b[col-1][i-1]
    b[col-1][0] = '_'
    # check win for player
    win = any(check_win_pop(b, col-1, i, player) for i in range(MAX_Y-1, -1, -1))
    # check if opponent also wins (loss for current player)
    opp = '@' if player == '#' else '#'
    loss = any(check_win_pop(b, col-1, i, opp) for i in range(MAX_Y-1, -1, -1))
    return b, win, loss

def _run(board, x, y, dx, dy, player):
    count = 0
    while 0 <= x < MAX_X and 0 <= y < MAX_Y and board[x][y] == player:
        count += 1
        x += dx; y += dy
    return count

def check_win_insert(board, x, y, player):
    if board[x][y] != player: return False
    dirs = [(1,0),(-1,0),(0,1),(0,-1),(1,1),(-1,-1),(1,-1),(-1,1)]
    for dx, dy in [(1,0),(0,1),(1,1),(1,-1)]:
        total = 1 + _run(board,x+dx,y+dy,dx,dy,player) + _run(board,x-dx,y-dy,-dx,-dy,player)
        if total >= 4: return True
    return False

def check_win_pop(board, x, y, player):
    if board[x][y] != player: return False
    for dx, dy in [(1,0),(1,1),(1,-1)]:
        total = 1 + _run(board,x+dx,y+dy,dx,dy,player) + _run(board,x-dx,y-dy,-dx,-dy,player)
        if total >= 4: return True
    return False

def legal_moves(board, player):
    """Return list of (col 1-based, option 0=insert/1=pop)."""
    moves = []
    for col in range(1, MAX_X+1):
        if board[col-1][0] == '_':            # can insert
            moves.append((col, 0))
        if board[col-1][MAX_Y-1] == player:   # can pop own piece
            moves.append((col, 1))
    return moves

print("Helpers loaded.")

## 3 — Generate training data via MCTS self-play
Each game is played MCTS vs MCTS. Every board state + chosen move is recorded.

In [ ]:
import csv, random

SELFPLAY_GAMES  = 30    # increase for a stronger ID3 model (takes longer)
MCTS_ITERS_DATA = 200   # iterations per move during data generation
CSV_PATH        = "bata_dasa.csv"
MAX_MOVES       = 200   # hard cap to prevent infinite games

def play_game_mcts(iterations=MCTS_ITERS_DATA):
    """Play one full MCTS self-play game. Returns list of (jogada, board_str, player)."""
    board   = empty_board()
    player  = '#'
    records = []

    for _ in range(MAX_MOVES):
        moves = legal_moves(board, player)
        if not moves:
            records.append(("draw", board_to_str(board), player))
            break

        col, opt = ask_mcts(board, player, iterations)

        if col == 0:
            records.append(("draw", board_to_str(board), player))
            break

        jogada = f"{col}-{'i' if opt == 0 else 'p'}"
        records.append((jogada, board_to_str(board), player))

        if opt == 0:
            board, win = do_insert(board, col, player)
        else:
            board, win, loss = do_pop(board, col, player)
            if loss: break   # current player's pop caused opponent to win

        if win:
            break

        player = '@' if player == '#' else '#'

    return records

print(f"Generating {SELFPLAY_GAMES} self-play games …")
all_records = []
for g in range(SELFPLAY_GAMES):
    recs = play_game_mcts()
    all_records.extend(recs)
    print(f"  game {g+1:3d}/{SELFPLAY_GAMES}  ({len(recs)} moves)", end="\r")

print(f"\nTotal records: {len(all_records)}")

with open(CSV_PATH, 'w', newline='') as f:
    w = csv.writer(f)
    for jogada, board_str, pl in all_records:
        w.writerow([jogada, board_str, pl])

print(f"Saved to {CSV_PATH}")

## 4 — Train the ID3 model

In [ ]:
import importlib.util, sys

spec = importlib.util.spec_from_file_location("id3", "ID3_DECISION_TREES.py")
id3_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(id3_mod)

tree, test_df, preds, acc = id3_mod.build_and_evaluate(CSV_PATH)

print(f"ID3 test accuracy: {acc*100:.1f}%")
default_class = test_df['Class'].value_counts().idxmax()

## 5 — ID3 move function

In [ ]:
def board_to_features(board, player):
    """Convert board + player into the feature dict the ID3 tree expects."""
    flat = board_to_str(board)
    row = {f"c{i}": flat[i] for i in range(42)}
    row["player"] = player
    return row

def ask_id3(board, player):
    """
    Query the ID3 tree for a move.
    Returns (col 1-based, option 0/1) or (0, 0) for draw.
    Falls back to a random legal move if the tree returns something illegal.
    """
    sample = board_to_features(board, player)
    prediction = id3_mod.predict_sample(tree, sample, default=default_class)

    # parse prediction string e.g. "4-i", "3-p", "draw"
    if prediction == "draw" or prediction is None:
        return 0, 0

    try:
        col_str, opt_str = prediction.split("-")
        col = int(col_str)
        opt = 0 if opt_str == 'i' else 1
    except Exception:
        col, opt = 0, 0

    # validate move is actually legal; if not, pick a random legal move
    moves = legal_moves(board, player)
    if (col, opt) not in moves:
        if not moves:
            return 0, 0
        col, opt = random.choice(moves)

    return col, opt

print("ID3 move function ready.")

## 6 — Play MCTS (`#`) vs ID3 (`@`)

In [ ]:
NUM_GAMES       = 20    # number of head-to-head games
MCTS_ITERS_PLAY = 500   # stronger than data-gen

results = []   # 'mcts', 'id3', or 'draw'

def play_match(mcts_player='#', id3_player='@',
               mcts_iters=MCTS_ITERS_PLAY):
    board  = empty_board()
    player = '#'   # # always goes first

    for move_num in range(MAX_MOVES):
        moves = legal_moves(board, player)
        if not moves:
            return 'draw'

        if player == mcts_player:
            col, opt = ask_mcts(board, player, mcts_iters)
        else:
            col, opt = ask_id3(board, player)

        if col == 0:
            return 'draw'

        if opt == 0:
            board, win = do_insert(board, col, player)
            if win:
                return 'mcts' if player == mcts_player else 'id3'
        else:
            board, win, loss = do_pop(board, col, player)
            if win:
                return 'mcts' if player == mcts_player else 'id3'
            if loss:
                # popping piece caused opponent to win
                return 'id3' if player == mcts_player else 'mcts'

        player = '@' if player == '#' else '#'

    return 'draw'

print(f"Playing {NUM_GAMES} games (MCTS as '#', ID3 as '@') …")
for g in range(NUM_GAMES):
    outcome = play_match()
    results.append(outcome)
    print(f"  game {g+1:3d}/{NUM_GAMES}  → {outcome}", end="\r")

print(f"\nDone. {NUM_GAMES} games played.")

## 7 — Results

In [ ]:
from collections import Counter

counts = Counter(results)
mcts_w = counts.get('mcts', 0)
id3_w  = counts.get('id3',  0)
draws  = counts.get('draw', 0)
total  = len(results)

print("=" * 40)
print(f"  MCTS  wins : {mcts_w:3d}  ({mcts_w/total*100:.1f}%)")
print(f"  ID3   wins : {id3_w:3d}  ({id3_w/total*100:.1f}%)")
print(f"  Draws      : {draws:3d}  ({draws/total*100:.1f}%)")
print("=" * 40)

# simple bar chart
try:
    import matplotlib.pyplot as plt
    labels = ['MCTS (#)', 'ID3 (@)', 'Draw']
    values = [mcts_w, id3_w, draws]
    colors = ['#4e79a7', '#f28e2b', '#76b7b2']
    plt.figure(figsize=(6, 4))
    bars = plt.bar(labels, values, color=colors, edgecolor='black')
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(val), ha='center', fontweight='bold')
    plt.title(f'MCTS vs ID3 — {total} games')
    plt.ylabel('Games won')
    plt.ylim(0, max(values) + 2)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not available — skipping chart)")